In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
import numpy as np
import plotly.express as px
from src.amp_colabfold.features import (
    compute_features_batch, feature_summary, PROCESSED_DIR
)

# load curated sequences from Stage 1
df = pd.read_csv(PROCESSED_DIR / "curated_amps_metadata.csv")
print(f"Loaded {len(df)} sequences")
print(df.columns.tolist())

Loaded 12435 sequences
['amp_id', 'id', 'name', 'sequence', 'activity', 'source']


In [2]:
print("Computing physicochemical features ...")
feat_df = compute_features_batch(df, seq_col="sequence", show_progress=True)
print(f"\nFeature matrix shape: {feat_df.shape}")
feat_df.head()

Computing physicochemical features ...
  Computing features: 12435/12435 ... done      

Feature matrix shape: (12435, 17)


,amp_id,length,net_charge_pH7,isoelectric_point,hydrophobicity_eisenberg,hydrophobic_moment,amphipathicity,instability_index,aromaticity,fraction_positive,fraction_negative,fraction_helix,fraction_sheet,fraction_turn,aliphatic_index,boman_index,molecular_weight
0,AMP_00000,50,2.116901,9.425624,0.1256,0.662274,0.662274,-0.340,0.12,0.08,0.04,0.44,0.42,0.48,66.2,1.0108,5327.98394
1,AMP_00001,50,2.684962,7.953788,-0.0128,0.403256,0.403256,35.946,0.10,0.10,0.04,0.38,0.44,0.58,48.8,1.4748,5499.28494
2,AMP_00002,50,13.085691,12.710738,-0.2630,0.833131,0.833131,39.280,0.06,0.26,0.00,0.44,0.28,0.44,60.6,2.6016,5294.12614
3,AMP_00003,50,5.806944,9.235209,0.0080,0.520710,0.520710,30.620,0.08,0.14,0.02,0.34,0.42,0.56,68.2,1.3096,5480.51494
4,AMP_00004,50,5.630372,9.100972,-0.3896,0.577204,0.577204,42.034,0.10,0.24,0.12,0.42,0.42,0.48,50.8,3.4208,5988.91454


In [3]:
summary = feature_summary(feat_df)
print("Feature summary:")
summary

Feature summary:


,mean,std,min,max,pct_missing
length,22.5860,8.9567,10.0000,50.0000,0.0
net_charge_pH7,2.1415,3.4031,-19.9908,35.9909,0.0
isoelectric_point,8.8431,2.7877,2.1846,13.8418,0.0
hydrophobicity_eisenberg,-0.0258,0.3043,-2.5300,0.9445,0.0
hydrophobic_moment,0.4846,0.1980,0.0128,1.2306,0.0
amphipathicity,0.4846,0.1980,0.0128,1.2306,0.0
instability_index,44.1429,37.6362,-73.3396,543.9467,0.0
aromaticity,0.1075,0.0847,0.0000,0.6364,0.0
fraction_positive,0.1584,0.1295,0.0000,1.0000,0.0
fraction_negative,0.0556,0.0658,0.0000,1.0000,0.0


In [4]:
# merge features back onto metadata
df_full = df.merge(feat_df, on="amp_id", how="left")
out_path = PROCESSED_DIR / "curated_amps_features.csv"
df_full.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
print(f"Shape: {df_full.shape}")
df_full.head(3)

Saved → D:\Github_projects\amp-colabfold\data\processed\curated_amps_features.csv
Shape: (12435, 22)


,amp_id,id,name,sequence,activity,source,length,net_charge_pH7,isoelectric_point,hydrophobicity_eisenberg,...,instability_index,aromaticity,fraction_positive,fraction_negative,fraction_helix,fraction_sheet,fraction_turn,aliphatic_index,boman_index,molecular_weight
0,AMP_00000,DRAMP00069,DRAMP00069,EYHLMNGANGYLTRVNGKTVYRVTKDPVSAVFGVISNCWGSAGAGF...,general,DRAMP4.0,50,2.116901,9.425624,0.1256,...,-0.340,0.12,0.08,0.04,0.44,0.42,0.48,66.2,1.0108,5327.98394
1,AMP_00001,DRAMP00416,DRAMP00416,KLCERSSGTWSGVCGNNNACKNQCIRLEGAQHGSCNYVFPAHKCIC...,general,DRAMP4.0,50,2.684962,7.953788,-0.0128,...,35.946,0.10,0.10,0.04,0.38,0.44,0.58,48.8,1.4748,5499.28494
2,AMP_00002,DRAMP18380,DRAMP18380,SGRGKTGGKARAKAKTRSSRAGLQFPVGRVHRLLRKGNYAQRVGAG...,general,DRAMP4.0,50,13.085691,12.710738,-0.2630,...,39.280,0.06,0.26,0.00,0.44,0.28,0.44,60.6,2.6016,5294.12614


In [5]:
fig = px.histogram(
    df_full,
    x="length",
    nbins=40,
    color="activity",
    title="AMP length distribution by activity class",
    labels={"length": "Peptide length (aa)", "count": "Count"},
    template="simple_white",
    opacity=0.8,
)
fig.update_layout(bargap=0.05)
fig.show()

In [6]:
fig = px.scatter(
    df_full.sample(min(3000, len(df_full)), random_state=42),
    x="net_charge_pH7",
    y="hydrophobic_moment",
    color="activity",
    hover_data=["amp_id", "length"],
    title="Net charge vs hydrophobic moment",
    labels={
        "net_charge_pH7": "Net charge (pH 7)",
        "hydrophobic_moment": "Hydrophobic moment",
    },
    template="simple_white",
    opacity=0.6,
)
fig.show()

In [7]:
import plotly.figure_factory as ff

feature_cols = [
    "length", "net_charge_pH7", "hydrophobicity_eisenberg",
    "hydrophobic_moment", "instability_index", "aromaticity",
    "fraction_positive", "fraction_negative",
    "fraction_helix", "fraction_sheet", "aliphatic_index",
    "boman_index", "molecular_weight",
]

corr = df_full[feature_cols].corr().round(2)
fig = ff.create_annotated_heatmap(
    z=corr.values,
    x=feature_cols,
    y=feature_cols,
    colorscale="RdBu",
    reversescale=True,
    showscale=True,
)
fig.update_layout(
    title="Feature correlation matrix",
    width=800, height=700,
)
fig.show()